[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srini2k03/jobpost-extractor-qlora/blob/main/notebooks/03_train_qlora.ipynb)

# 03 · QLoRA fine-tuning (TRL SFTTrainer)

Fine-tunes **Qwen2.5-1.5B-Instruct** with QLoRA (4-bit NF4, double quantisation, fp16 compute), LoRA r=16/α=32 on all linear projections, completion-only loss, gradient checkpointing, cosine LR 2e-4 with 3% warmup, effective batch 16, early stopping on validation JSON micro F1. Weights & Biases logs loss, LR, eval metrics and sample generations. Optional: the small r/lr sweep behind `docs/hyperparameters.md`, and the Llama-3.2-1B comparison run.

**How to run**
1. Runtime → Change runtime type → **T4 GPU**.
2. Secrets (key icon in the left sidebar), each with *Notebook access* switched on: `HF_TOKEN`, `HF_USERNAME`, `WANDB_API_KEY`.
   `HF_USERNAME` is your Hugging Face account name; `HF_TOKEN` needs *write* access.
3. Runtime → **Run all**. Expected runtime: **~1–1.5 h for the main run; sweep +~25 min per run**. Cells after the setup are safe to re-run.

**Outputs:** LoRA adapter pushed to `<HF_USERNAME>/jobx-qwen2.5-1.5b-qlora` (with model card); `results/training/<run>/log_history.json`, `best.json`, `curves.png` pushed to `<HF_USERNAME>/jobx-artifacts`; the W&B run.

All code lives in `src/jobx` of the GitHub repo; this notebook only orchestrates it.

In [ ]:
# Setup: clone the repo (code + data), install pinned dependencies, install jobx itself.
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/srini2k03/jobpost-extractor-qlora.git"
if IN_COLAB:
    if not os.path.exists("/content/jobpost-extractor-qlora"):
        !git clone --depth 1 {REPO_URL} /content/jobpost-extractor-qlora
    %cd /content/jobpost-extractor-qlora
    !pip install -q transformers==4.56.2 peft==0.17.1 trl==0.23.1 bitsandbytes==0.48.1 accelerate==1.10.1 datasets==4.1.1 huggingface_hub==0.35.3 wandb==0.22.2 lm-format-enforcer==0.11.3
    !pip install -q -e .
else:
    # running locally from a checkout: notebooks/ -> repo root
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
# Secrets -> environment variables (read by jobx.hub and wandb).
import os
for key in ("HF_TOKEN", "HF_USERNAME", "WANDB_API_KEY"):
    if not os.environ.get(key):
        try:
            from google.colab import userdata
            os.environ[key] = userdata.get(key)
        except Exception:
            print(f"secret {key} not available")
from jobx.hub import hf_username, repo_id
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Hub account:", hf_username(), "| artifacts repo:", repo_id("artifacts"))

In [ ]:
CONFIG = 'configs/train/qwen2.5-1.5b-qlora.yaml'
RUN_SWEEP = False
RUN_LLAMA = False   # gated: accept the Llama 3.2 licence on HF first
import json
from pathlib import Path
from jobx.train.config import load_train_config, sweep_runs
from jobx.train.sft import train
cfg = load_train_config(CONFIG)
print(json.dumps(cfg, indent=2))

## Train

In [ ]:
final = train(cfg)
print(Path(final, 'best.json').read_text())

## Curves + push artefacts

In [ ]:
import os, shutil
from jobx.eval.curves import plot_curves
from jobx.baselines.runs import push_results
from jobx.cards import render_card, metrics_table
from jobx.hub import repo_id, push_folder
run_dir = Path('results/training') / cfg['run_name']
run_dir.mkdir(parents=True, exist_ok=True)
for f in ('log_history.json', 'best.json', 'train_config.json'):
    shutil.copy(Path(final, f), run_dir / f)
png = plot_curves(run_dir / 'log_history.json', run_dir / 'curves.png')
shutil.copy(png, 'results/training/curves.png')
push_results([run_dir / 'log_history.json', run_dir / 'best.json', run_dir / 'train_config.json', png.resolve(), Path('results/training/curves.png').resolve()])
import wandb
card = render_card('adapter', {
    'base_model': cfg['base_model'], 'repo_name': repo_id('adapter'), 'artifacts_repo': repo_id('artifacts'),
    'n_train': sum(1 for _ in open(cfg['data']['train'])), 'lora_r': cfg['lora']['r'], 'lora_alpha': cfg['lora']['alpha'],
    'learning_rate': cfg['training']['learning_rate'], 'wandb_url': os.environ.get('WANDB_RUN_URL', 'see W&B project jobx-qlora'),
    'metrics_table': metrics_table(['qwen2.5-1.5b_qlora']), 'base_licence': 'Apache-2.0'})
Path(final, 'README.md').write_text(card)
print('pushed adapter to', push_folder(final, 'adapter', commit_message='QLoRA adapter'))

## Optional: hyper-parameter sweep (r ∈ {8,16,32}, lr ∈ {1e-4, 2e-4}; 1 epoch each)

In [ ]:
if RUN_SWEEP:
    summary = {}
    for name, scfg in sweep_runs('configs/train/sweep.yaml'):
        out = train(scfg)
        summary[name] = json.loads(Path(out, 'best.json').read_text())
        print(name, summary[name])
    Path('results/training/sweep_summary.json').write_text(json.dumps(summary, indent=2))
    push_results([Path('results/training/sweep_summary.json').resolve()])

## Optional: Llama-3.2-1B-Instruct with the same recipe

In [ ]:
if RUN_LLAMA:
    lcfg = load_train_config('configs/train/llama3.2-1b-qlora.yaml')
    lfinal = train(lcfg)
    print('pushed', push_folder(lfinal, 'adapter_llama', commit_message='QLoRA adapter (Llama 3.2 1B)'))